## A.I. Assignment 5

## Learning Goals

By the end of this lab, you should be able to:
* Get more familiar with tensors in pytorch 
* Create a simple multilayer perceptron model with pytorch
* Visualise the parameters


### Task

Build a fully connected feed forward network that adds two bits. Determine the a propper achitecture for this network (what database you use for this problem? how many layers? how many neurons on each layer? what is the activation function? what is the loss function? etc)

Create at least 3 such networks and compare their performance (how accurate they are?, how farst they are trained to get at 1 accuracy?)

Display for the best one the weights for each layer.


In [13]:
import time
import torch
import torch.nn as nn
from collections import OrderedDict


In [17]:
# Define at least 3 different fully connected models
model1 = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(2, 4)),
    ('act1', nn.Tanh()),
    ('fc2', nn.Linear(4, 2))
]))

model2 = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(2, 8)),
    ('act1', nn.ReLU()),
    ('fc2', nn.Linear(8, 2))
]))

model3 = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(2, 6)),
    ('act1', nn.Tanh()),
    ('fc2', nn.Linear(6, 4)),
    ('act2', nn.Tanh()),
    ('fc3', nn.Linear(4, 2))
]))

models = {
    'model1_tanh_2_4_2': model1,
    'model2_relu_2_8_2': model2,
    'model3_tanh_2_6_4_2': model3
}

In [18]:
for name, model in models.items():
    print(f"\n{name}")
    print(model)


model1_tanh_2_4_2
Sequential(
  (fc1): Linear(in_features=2, out_features=4, bias=True)
  (act1): Tanh()
  (fc2): Linear(in_features=4, out_features=2, bias=True)
)

model2_relu_2_8_2
Sequential(
  (fc1): Linear(in_features=2, out_features=8, bias=True)
  (act1): ReLU()
  (fc2): Linear(in_features=8, out_features=2, bias=True)
)

model3_tanh_2_6_4_2
Sequential(
  (fc1): Linear(in_features=2, out_features=6, bias=True)
  (act1): Tanh()
  (fc2): Linear(in_features=6, out_features=4, bias=True)
  (act2): Tanh()
  (fc3): Linear(in_features=4, out_features=2, bias=True)
)


In [19]:
# your code here
data_in = torch.tensor([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
])

print(data_in)

tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])


In [20]:
# your code here
data_target = torch.tensor([
    [0., 0.],  # 0+0=0
    [1., 0.],  # 0+1=1
    [1., 0.],  # 1+0=1
    [0., 1.]   # 1+1=2 => binary 10 => sum=0, carry=1
])
print(data_target)

tensor([[0., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.]])


In [21]:
criterion = nn.BCEWithLogitsLoss()
max_epochs = 1000

In [22]:
# Train all models and collect comparison metrics
def train_and_evaluate(model, data_in, data_target, criterion, max_epochs=1000, lr=0.1):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    epoch_to_perfect = None
    losses = []

    start = time.perf_counter()
    model.train()
    for epoch in range(max_epochs):
        optimizer.zero_grad()
        logits = model(data_in)
        loss = criterion(logits, data_target)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Evaluation
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            sample_correct = (preds == data_target).all(dim=1).float()
            acc = sample_correct.mean().item()

        if epoch_to_perfect is None and acc == 1.0:
            epoch_to_perfect = epoch + 1

    elapsed = time.perf_counter() - start

    model.eval()
    # Evaluation
    with torch.no_grad():
        final_logits = model(data_in)
        final_probs = torch.sigmoid(final_logits)
        final_preds = (final_probs >= 0.5).float()
        final_acc = (final_preds == data_target).all(dim=1).float().mean().item()

    return {
        'epoch_to_perfect': epoch_to_perfect,
        'final_accuracy': final_acc,
        'train_time_sec': elapsed,
        'losses': losses,
        'final_probs': final_probs,
        'final_preds': final_preds
    }

results = {}
for name, model in models.items():
    results[name] = train_and_evaluate(
        model, data_in, data_target, criterion, max_epochs=max_epochs, lr=0.1
    )

# Pick best by: highest final accuracy, then lowest epoch to perfect, then shortest time
best_model_name = min(
    results.keys(),
    key=lambda n: (
        -results[n]['final_accuracy'],
        results[n]['epoch_to_perfect'] if results[n]['epoch_to_perfect'] is not None else 10**9,
        results[n]['train_time_sec']
    )
)
best_model = models[best_model_name]

In [23]:
# Visualize and compare results
print('Comparison of models:')
for name, info in results.items():
    print(f"\n{name}")
    print(f"  Final accuracy: {info['final_accuracy']:.2f}")
    print(f"  Epoch to 1.00 accuracy: {info['epoch_to_perfect']}")
    print(f"  Train time (s): {info['train_time_sec']:.6f}")

print(f"\nBest model: {best_model_name}")

print('\nPredictions of best model:')
best_info = results[best_model_name]
print('Input:')
print(data_in)
print('Target [sum, carry]:')
print(data_target)
print('Predicted probabilities:')
print(best_info['final_probs'])
print('Predicted bits:')
print(best_info['final_preds'])

Comparison of models:

model1_tanh_2_4_2
  Final accuracy: 0.50
  Epoch to 1.00 accuracy: None
  Train time (s): 0.520431

model2_relu_2_8_2
  Final accuracy: 1.00
  Epoch to 1.00 accuracy: 576
  Train time (s): 0.595346

model3_tanh_2_6_4_2
  Final accuracy: 0.75
  Epoch to 1.00 accuracy: None
  Train time (s): 0.687873

Best model: model2_relu_2_8_2

Predictions of best model:
Input:
tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
Target [sum, carry]:
tensor([[0., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.]])
Predicted probabilities:
tensor([[0.1446, 0.0031],
        [0.9506, 0.0062],
        [0.7959, 0.0849],
        [0.0769, 0.9705]])
Predicted bits:
tensor([[0., 0.],
        [1., 0.],
        [1., 0.],
        [0., 1.]])


In [16]:
# Print weights of the best model
print(f"Best model weights: {best_model_name}")
for layer_name, param in best_model.named_parameters():
    print(f"\n{layer_name}:")
    print(param.data)

Best model weights: model2_relu_2_8_2

fc1.weight:
tensor([[ 0.0611, -0.0557],
        [ 2.1385,  2.0712],
        [-0.4717, -0.0992],
        [-0.2253, -0.0068],
        [-0.3320, -0.7611],
        [-0.5971,  0.5267],
        [-0.2043, -0.4835],
        [-1.8884, -1.8899]])

fc1.bias:
tensor([-0.1755, -2.0737, -0.2017, -0.0115,  1.5838, -0.5637,  1.2828,  1.8862])

fc2.weight:
tensor([[ 0.0472, -2.1021, -0.0234, -0.0544,  0.9407,  0.0994,  0.6098, -3.1308],
        [-0.2602,  2.8887, -0.2183,  0.0122, -1.3494,  0.0661, -0.9888, -0.4557]])

fc2.bias:
tensor([ 0.9296, -1.6804])
